## Overview

This notebook constructs the feature engineering to turn a basic crop prediction model into a comprehensive agricultural intelligence system that transforms environmental and soil data into actionable insights. We are using the Crop Recommendation Dataset, available on Kaggle. This code processes soil nutrient levels (N, P, K) and climate data to support the project's two core capabilities: recommending ideal crops based on environmental conditions and optimizing fertilizer use by identifying critical nutrient deficiencies and imbalances

In [ ]:
!pip install pandas numpy scikit-learn kagglehub

In [3]:
import pandas as pd
import numpy as np
import kagglehub
import os
from google.colab import drive

In [4]:
# Import dataset
print("Downloading dataset from Kaggle...")
path = kagglehub.dataset_download("atharvaingle/crop-recommendation-dataset")

csv_file = [f for f in os.listdir(path) if f.endswith('.csv')][0]
full_path = os.path.join(path, csv_file)
df = pd.read_csv(full_path)
print(f"Data successfully loaded! Shape: {df.shape}")

Using Colab cache for faster access to the 'crop-recommendation-dataset' dataset.
Data successfully loaded! Shape: (2200, 8)


## Feature engineering

In [5]:
# Heat & humidity index to approximates combined stress on the crop
df['heat_humidity_index'] = (df['temperature'] * df['humidity']) / 100

# Water availability proxy for ratio of rainfall to temperature (+0.1 to avoid division by zero)
df['water_availability_index'] = df['rainfall'] / (df['temperature'] + 0.1)

# pH Classification
conditions = [
    (df['ph'] < 5.5),
    (df['ph'] >= 5.5) & (df['ph'] <= 6.5),
    (df['ph'] > 6.5) & (df['ph'] <= 7.5),
    (df['ph'] > 7.5)
]
choices = ['Strongly Acidic', 'Slightly Acidic', 'Neutral', 'Alkaline']
df['ph_category'] = np.select(conditions, choices, default='Unknown')

# Basic nutrient ratios
df['N_P_ratio'] = df['N'] / (df['P'] + 0.1)
df['N_K_ratio'] = df['N'] / (df['K'] + 0.1)
df['P_K_ratio'] = df['P'] / (df['K'] + 0.1)
df['total_NPK'] = df['N'] + df['P'] + df['K']

# Calculate ideal mean N, P, K for each crop label
crop_profiles = df.groupby('label')[['N', 'P', 'K']].mean().reset_index()
crop_profiles.rename(columns={'N': 'ideal_N', 'P': 'ideal_P', 'K': 'ideal_K'}, inplace=True)

# Merge the ideal profiles back into the main data
df = pd.merge(df, crop_profiles, on='label', how='left')

# Calculate adjustments: negative = deficiency, positive = excess
df['N_adjustment_req'] = df['ideal_N'] - df['N']
df['P_adjustment_req'] = df['ideal_P'] - df['P']
df['K_adjustment_req'] = df['ideal_K'] - df['K']
df['severe_N_deficiency'] = (df['N_adjustment_req'] > 20).astype(int)
df['severe_P_deficiency'] = (df['P_adjustment_req'] > 20).astype(int)
df['severe_K_deficiency'] = (df['K_adjustment_req'] > 20).astype(int)

In [6]:
print(df[['label', 'temperature', 'heat_humidity_index', 'ph_category',
          'N_adjustment_req', 'severe_N_deficiency']].head())

  label  temperature  heat_humidity_index ph_category  N_adjustment_req  \
0  rice    20.879744            17.121963     Neutral            -10.11   
1  rice    21.770462            17.485957     Neutral             -5.11   
2  rice    23.004459            18.937446    Alkaline             19.89   
3  rice    26.491096            21.234829     Neutral              5.89   
4  rice    20.130175            16.427204    Alkaline              1.89   

   severe_N_deficiency  
0                    0  
1                    0  
2                    0  
3                    0  
4                    0  


## Export

In [7]:
drive.mount('/content/drive')

drive_folder = '/content/drive/My Drive/ag_project'

# Create the folder if it doesn't already exist
if not os.path.exists(drive_folder):
    os.makedirs(drive_folder)
    print(f"Created new folder at: {drive_folder}")

output_filepath = os.path.join(drive_folder, 'engineered_ag_data.csv')

df.to_csv(output_filepath, index=False)
print(f"\nSaved to Google Drive at: {output_filepath}")

Mounted at /content/drive

Saved to Google Drive at: /content/drive/My Drive/ag_project/engineered_ag_data.csv
